<a href="https://colab.research.google.com/github/evildead23151/SparseRNN-Systems/blob/main/v3_sparseRNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install ninja

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 13.8 MB/s eta 0:00:00


In [8]:
# =========================
# BLOCK 1: ENV + SANITY
# =========================

import os, torch

# Force stable env
os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
os.environ["MAX_JOBS"] = "1"

print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

# Check NVCC
!nvcc --version

# Check compiler
!gcc --version

# ABI check
abi = 1 if torch._C._GLIBCXX_USE_CXX11_ABI else 0
print("Using ABI:", abi)

Torch CUDA: 12.8
CUDA available: True
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

Using ABI: 1


In [9]:
# =========================
# BLOCK 2: MINIMAL CUDA TEST
# =========================

from torch.utils.cpp_extension import load
import time, os

os.system("rm -rf test_ext")
os.system("rm -rf /root/.cache/torch_extensions")
os.makedirs("test_ext", exist_ok=True)

# CUDA file
with open("test_ext/test.cu", "w") as f:
    f.write(r'''
#include <torch/extension.h>

__global__ void add_kernel(float* x) {
    int i = threadIdx.x;
    x[i] += 1.0f;
}

void launch(torch::Tensor x) {
    add_kernel<<<1, 64>>>(x.data_ptr<float>());
}
''')

# Binding
with open("test_ext/bind.cpp", "w") as f:
    f.write(r'''
#include <torch/extension.h>

void launch(torch::Tensor x);

void run(torch::Tensor x) {
    launch(x);
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("run", &run);
}
''')

# Compile
name = f"test_ext_{int(time.time())}"

mod = load(
    name=name,
    sources=["test_ext/bind.cpp", "test_ext/test.cu"],
    verbose=True,
    extra_cuda_cflags=["-O2"],
    extra_cflags=["-O2"]
)

print("✅ Minimal extension loaded")

✅ Minimal extension loaded


In [10]:
# =========================
# BLOCK 3: KERNEL SKELETON
# =========================

from torch.utils.cpp_extension import load
import os, time, torch

os.system("rm -rf v3")
os.system("rm -rf /root/.cache/torch_extensions")
os.makedirs("v3", exist_ok=True)

# =========================
# CUDA KERNEL (SIMPLE WRITE)
# =========================
with open("v3/sparse_kernel.cu", "w") as f:
    f.write(r'''
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

#define BLOCK_SIZE 64

__global__ void test_kernel(
    const float* x,
    float* out,
    int B, int T, int H
) {
    int b = blockIdx.x;
    int h = blockIdx.y * BLOCK_SIZE + threadIdx.x;

    if (b >= B || h >= H) return;

    for (int t = 0; t < T; t++) {
        int idx = b * T * H + t * H + h;

        // simple copy
        out[idx] = x[idx] * 2.0f;
    }
}

void launch_test(
    torch::Tensor x,
    torch::Tensor out
) {
    int B = x.size(0);
    int T = x.size(1);
    int H = x.size(2);

    int num_blocks = H / BLOCK_SIZE;

    dim3 grid(B, num_blocks);
    dim3 block(BLOCK_SIZE);

    test_kernel<<<grid, block>>>(
        x.data_ptr<float>(),
        out.data_ptr<float>(),
        B, T, H
    );
}
''')

# =========================
# BINDING
# =========================
with open("v3/binding.cpp", "w") as f:
    f.write(r'''
#include <torch/extension.h>

void launch_test(
    torch::Tensor x,
    torch::Tensor out
);

torch::Tensor forward(torch::Tensor x) {
    auto out = torch::zeros_like(x);
    launch_test(x, out);
    return out;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("forward", &forward);
}
''')

# =========================
# COMPILE
# =========================
name = f"v3_test_{int(time.time())}"

mod = load(
    name=name,
    sources=["v3/binding.cpp", "v3/sparse_kernel.cu"],
    verbose=True,
    extra_cuda_cflags=["-O2"],
    extra_cflags=["-O2"]
)

print("✅ BLOCK 3 compiled")

# =========================
# TEST
# =========================
x = torch.randn(64, 128, 512, device="cuda")

out = mod.forward(x)

print("Output shape:", out.shape)
print("Check:", torch.allclose(out, x * 2, atol=1e-5))

✅ BLOCK 3 compiled
Output shape: torch.Size([64, 128, 512])
Check: True


In [11]:
# =========================
# BLOCK 4: SHARED MEMORY TILE
# =========================

from torch.utils.cpp_extension import load
import os, time, torch

os.system("rm -rf v3")
os.system("rm -rf /root/.cache/torch_extensions")
os.makedirs("v3", exist_ok=True)

# =========================
# CUDA KERNEL
# =========================
with open("v3/sparse_kernel.cu", "w") as f:
    f.write(r'''
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

#define BLOCK_SIZE 64

__global__ void test_shared_kernel(
    const float* x,
    const float* W,
    float* out,
    int B, int T, int H, int num_blocks
) {
    int b = blockIdx.x;
    int block_id = blockIdx.y;
    int tid = threadIdx.x;

    int h = block_id * BLOCK_SIZE + tid;

    if (b >= B || h >= H) return;

    // --------------------------
    // SHARED MEMORY TILE
    // --------------------------
    __shared__ float W_shared[BLOCK_SIZE][BLOCK_SIZE];

    int total = BLOCK_SIZE * BLOCK_SIZE;

    // cooperative loading
    for (int i = tid; i < total; i += BLOCK_SIZE) {
        int r = i / BLOCK_SIZE;
        int c = i % BLOCK_SIZE;

        W_shared[r][c] =
            W[block_id * total + r * BLOCK_SIZE + c];
    }

    __syncthreads();

    // --------------------------
    // COMPUTE
    #pragma unroll
    for (int t = 0; t < T; t++) {

        float acc = 0.f;

        for (int k = 0; k < BLOCK_SIZE; k++) {
            float x_k = x[b * T * H + t * H + block_id * BLOCK_SIZE + k];
            acc += W_shared[tid][k] * x_k;
        }

        int idx = b * T * H + t * H + h;
        out[idx] = acc;
    }
}

void launch_test_shared(
    torch::Tensor x,
    torch::Tensor W,
    torch::Tensor out
) {
    int B = x.size(0);
    int T = x.size(1);
    int H = x.size(2);

    int num_blocks = H / BLOCK_SIZE;

    dim3 grid(B, num_blocks);
    dim3 block(BLOCK_SIZE);

    test_shared_kernel<<<grid, block>>>(
        x.data_ptr<float>(),
        W.data_ptr<float>(),
        out.data_ptr<float>(),
        B, T, H, num_blocks
    );
}
''')

# =========================
# BINDING
# =========================
with open("v3/binding.cpp", "w") as f:
    f.write(r'''
#include <torch/extension.h>

void launch_test_shared(
    torch::Tensor x,
    torch::Tensor W,
    torch::Tensor out
);

torch::Tensor forward(torch::Tensor x, torch::Tensor W) {
    auto out = torch::zeros_like(x);
    launch_test_shared(x, W, out);
    return out;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("forward", &forward);
}
''')

# =========================
# COMPILE
# =========================
name = f"v3_shared_{int(time.time())}"

mod = load(
    name=name,
    sources=["v3/binding.cpp", "v3/sparse_kernel.cu"],
    verbose=True,
    extra_cuda_cflags=["-O2"],
    extra_cflags=["-O2"]
)

print("✅ BLOCK 4 compiled")

# =========================
# TEST
# =========================
B, T, H = 64, 128, 512
x = torch.randn(B, T, H, device="cuda")

num_blocks = H // 64
W = torch.randn(num_blocks, 64, 64, device="cuda")

out = mod.forward(x, W)

print("Output shape:", out.shape)

✅ BLOCK 4 compiled
Output shape: torch.Size([64, 128, 512])


In [12]:
# =========================
# BLOCK 5: TIME FUSION + HIDDEN STATE
# =========================

from torch.utils.cpp_extension import load
import os, time, torch

os.system("rm -rf v3")
os.system("rm -rf /root/.cache/torch_extensions")
os.makedirs("v3", exist_ok=True)

# =========================
# CUDA KERNEL
# =========================
with open("v3/sparse_kernel.cu", "w") as f:
    f.write(r'''
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

#define BLOCK_SIZE 64

__global__ void test_time_fused_kernel(
    const float* x,
    const float* W,
    float* h,        // persistent hidden
    float* out,
    int B, int T, int H, int num_blocks
) {
    int b = blockIdx.x;
    int block_id = blockIdx.y;
    int tid = threadIdx.x;

    int h_idx = block_id * BLOCK_SIZE + tid;

    if (b >= B || h_idx >= H) return;

    // --------------------------
    // SHARED MEMORY (W)
    // --------------------------
    __shared__ float W_shared[BLOCK_SIZE][BLOCK_SIZE];

    int total = BLOCK_SIZE * BLOCK_SIZE;

    for (int i = tid; i < total; i += BLOCK_SIZE) {
        int r = i / BLOCK_SIZE;
        int c = i % BLOCK_SIZE;

        W_shared[r][c] =
            W[block_id * total + r * BLOCK_SIZE + c];
    }

    __syncthreads();

    // --------------------------
    // REGISTER STATE
    // --------------------------
    float h_val = h[b * H + h_idx];

    // --------------------------
    // TIME LOOP (FUSED)
    // --------------------------
    for (int t = 0; t < T; t++) {

        float acc = 0.f;

        // matrix multiply with hidden
        #pragma unroll
        for (int k = 0; k < BLOCK_SIZE; k++) {
            float h_k = h[b * H + block_id * BLOCK_SIZE + k];
            acc += W_shared[tid][k] * h_k;
        }

        // add input
        float x_val = x[b * T * H + t * H + h_idx];

        h_val = tanhf(acc + x_val);

        // write output
        out[b * T * H + t * H + h_idx] = h_val;

        // update global hidden (for next timestep)
        h[b * H + h_idx] = h_val;
    }
}

void launch_time_fused(
    torch::Tensor x,
    torch::Tensor W,
    torch::Tensor h,
    torch::Tensor out
) {
    int B = x.size(0);
    int T = x.size(1);
    int H = x.size(2);

    int num_blocks = H / BLOCK_SIZE;

    dim3 grid(B, num_blocks);
    dim3 block(BLOCK_SIZE);

    test_time_fused_kernel<<<grid, block>>>(
        x.data_ptr<float>(),
        W.data_ptr<float>(),
        h.data_ptr<float>(),
        out.data_ptr<float>(),
        B, T, H, num_blocks
    );
}
''')

# =========================
# BINDING
# =========================
with open("v3/binding.cpp", "w") as f:
    f.write(r'''
#include <torch/extension.h>

void launch_time_fused(
    torch::Tensor x,
    torch::Tensor W,
    torch::Tensor h,
    torch::Tensor out
);

torch::Tensor forward(torch::Tensor x, torch::Tensor W, torch::Tensor h) {
    auto out = torch::zeros_like(x);
    launch_time_fused(x, W, h, out);
    return out;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("forward", &forward);
}
''')

# =========================
# COMPILE
# =========================
name = f"v3_time_{int(time.time())}"

mod = load(
    name=name,
    sources=["v3/binding.cpp", "v3/sparse_kernel.cu"],
    verbose=True,
    extra_cuda_cflags=["-O2"],
    extra_cflags=["-O2"]
)

print("✅ BLOCK 5 compiled")

# =========================
# TEST
# =========================
B, T, H = 64, 128, 512

x = torch.randn(B, T, H, device="cuda")
W = torch.randn(H // 64, 64, 64, device="cuda")
h = torch.zeros(B, H, device="cuda")

out = mod.forward(x, W, h)

print("Output shape:", out.shape)

✅ BLOCK 5 compiled
Output shape: torch.Size([64, 128, 512])


In [13]:
# =========================
# BLOCK 6: REGISTER CACHING (h)
# =========================

from torch.utils.cpp_extension import load
import os, time, torch

os.system("rm -rf v3")
os.system("rm -rf /root/.cache/torch_extensions")
os.makedirs("v3", exist_ok=True)

# =========================
# CUDA KERNEL
# =========================
with open("v3/sparse_kernel.cu", "w") as f:
    f.write(r'''
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

#define BLOCK_SIZE 64

__global__ void test_register_kernel(
    const float* x,
    const float* W,
    float* h,
    float* out,
    int B, int T, int H, int num_blocks
) {
    int b = blockIdx.x;
    int block_id = blockIdx.y;
    int tid = threadIdx.x;

    int h_idx = block_id * BLOCK_SIZE + tid;

    if (b >= B || h_idx >= H) return;

    // --------------------------
    // SHARED MEMORY (W)
    // --------------------------
    __shared__ float W_shared[BLOCK_SIZE][BLOCK_SIZE];

    int total = BLOCK_SIZE * BLOCK_SIZE;

    for (int i = tid; i < total; i += BLOCK_SIZE) {
        int r = i / BLOCK_SIZE;
        int c = i % BLOCK_SIZE;

        W_shared[r][c] =
            W[block_id * total + r * BLOCK_SIZE + c];
    }

    __syncthreads();

    // --------------------------
    // REGISTER CACHE (h block)
    // --------------------------
    float h_reg[BLOCK_SIZE];

    #pragma unroll
    for (int k = 0; k < BLOCK_SIZE; k++) {
        h_reg[k] = h[b * H + block_id * BLOCK_SIZE + k];
    }

    float h_val = h_reg[tid];

    // --------------------------
    // TIME LOOP
    // --------------------------
    for (int t = 0; t < T; t++) {

        float acc = 0.f;

        #pragma unroll
        for (int k = 0; k < BLOCK_SIZE; k++) {
            acc += W_shared[tid][k] * h_reg[k];
        }

        float x_val = x[b * T * H + t * H + h_idx];

        h_val = tanhf(acc + x_val);

        // update register cache
        h_reg[tid] = h_val;

        // write output
        out[b * T * H + t * H + h_idx] = h_val;
    }

    // write back final hidden
    h[b * H + h_idx] = h_val;
}

void launch_register(
    torch::Tensor x,
    torch::Tensor W,
    torch::Tensor h,
    torch::Tensor out
) {
    int B = x.size(0);
    int T = x.size(1);
    int H = x.size(2);

    int num_blocks = H / BLOCK_SIZE;

    dim3 grid(B, num_blocks);
    dim3 block(BLOCK_SIZE);

    test_register_kernel<<<grid, block>>>(
        x.data_ptr<float>(),
        W.data_ptr<float>(),
        h.data_ptr<float>(),
        out.data_ptr<float>(),
        B, T, H, num_blocks
    );
}
''')

# =========================
# BINDING
# =========================
with open("v3/binding.cpp", "w") as f:
    f.write(r'''
#include <torch/extension.h>

void launch_register(
    torch::Tensor x,
    torch::Tensor W,
    torch::Tensor h,
    torch::Tensor out
);

torch::Tensor forward(torch::Tensor x, torch::Tensor W, torch::Tensor h) {
    auto out = torch::zeros_like(x);
    launch_register(x, W, h, out);
    return out;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("forward", &forward);
}
''')

# =========================
# COMPILE
# =========================
name = f"v3_register_{int(time.time())}"

mod = load(
    name=name,
    sources=["v3/binding.cpp", "v3/sparse_kernel.cu"],
    verbose=True,
    extra_cuda_cflags=["-O2"],
    extra_cflags=["-O2"]
)

print("✅ BLOCK 6 compiled")

# =========================
# TEST
# =========================
B, T, H = 64, 128, 512

x = torch.randn(B, T, H, device="cuda")
W = torch.randn(H // 64, 64, 64, device="cuda")
h = torch.zeros(B, H, device="cuda")

out = mod.forward(x, W, h)

print("Output shape:", out.shape)

✅ BLOCK 6 compiled
Output shape: torch.Size([64, 128, 512])


In [16]:
# =========================
# BLOCK 7: FULL LSTM (STABLE VERSION)
# =========================

from torch.utils.cpp_extension import load
import os, time, torch

os.system("rm -rf v3")
os.system("rm -rf /root/.cache/torch_extensions")
os.makedirs("v3", exist_ok=True)

# =========================
# CUDA KERNEL
# =========================
with open("v3/sparse_kernel.cu", "w") as f:
    f.write(r'''
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

#define BLOCK_SIZE 64

__device__ inline float sigmoidf(float x) {
    return 1.f / (1.f + expf(-x));
}

__global__ void lstm_kernel(
    const float* x,
    const float* W,
    float* h,
    float* c,
    float* out,
    int B, int T, int H
) {
    int b = blockIdx.x;
    int block_id = blockIdx.y;
    int tid = threadIdx.x;

    int h_idx = block_id * BLOCK_SIZE + tid;

    if (b >= B || h_idx >= H) return;

    __shared__ float W_shared[BLOCK_SIZE][BLOCK_SIZE];

    int total = BLOCK_SIZE * BLOCK_SIZE;

    float h_val = h[b * H + h_idx];
    float c_val = c[b * H + h_idx];

    for (int t = 0; t < T; t++) {

        float i_val = 0.f, f_val = 0.f, g_val = 0.f, o_val = 0.f;

        for (int g = 0; g < 4; g++) {

            // load one gate at a time
            for (int i = tid; i < total; i += BLOCK_SIZE) {
                int r = i / BLOCK_SIZE;
                int c_idx = i % BLOCK_SIZE;

                W_shared[r][c_idx] =
                    W[block_id * 4 * total +
                      g * total +
                      r * BLOCK_SIZE + c_idx];
            }

            __syncthreads();

            float acc = 0.f;

            for (int k = 0; k < BLOCK_SIZE; k++) {
                float h_k = h[b * H + block_id * BLOCK_SIZE + k];
                acc += W_shared[tid][k] * h_k;
            }

            float x_val = x[b * T * H + t * H + h_idx];

            if (g == 0) i_val = sigmoidf(acc + x_val);
            else if (g == 1) f_val = sigmoidf(acc + x_val);
            else if (g == 2) g_val = tanhf(acc + x_val);
            else o_val = sigmoidf(acc + x_val);

            __syncthreads();
        }

        c_val = f_val * c_val + i_val * g_val;
        h_val = o_val * tanhf(c_val);

        out[b * T * H + t * H + h_idx] = h_val;

        h[b * H + h_idx] = h_val;
    }

    c[b * H + h_idx] = c_val;
}

void launch_lstm(
    torch::Tensor x,
    torch::Tensor W,
    torch::Tensor h,
    torch::Tensor c,
    torch::Tensor out
) {
    int B = x.size(0);
    int T = x.size(1);
    int H = x.size(2);

    int num_blocks = H / BLOCK_SIZE;

    dim3 grid(B, num_blocks);
    dim3 block(BLOCK_SIZE);

    lstm_kernel<<<grid, block>>>(
        x.data_ptr<float>(),
        W.data_ptr<float>(),
        h.data_ptr<float>(),
        c.data_ptr<float>(),
        out.data_ptr<float>(),
        B, T, H
    );
}
''')

# =========================
# BINDING
# =========================
with open("v3/binding.cpp", "w") as f:
    f.write(r'''
#include <torch/extension.h>

void launch_lstm(
    torch::Tensor x,
    torch::Tensor W,
    torch::Tensor h,
    torch::Tensor c,
    torch::Tensor out
);

torch::Tensor forward(
    torch::Tensor x,
    torch::Tensor W,
    torch::Tensor h,
    torch::Tensor c
) {
    auto out = torch::zeros_like(x);
    launch_lstm(x, W, h, c, out);
    return out;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("forward", &forward);
}
''')

# =========================
# COMPILE
# =========================
name = f"v3_lstm_{int(time.time())}"

mod = load(
    name=name,
    sources=["v3/binding.cpp", "v3/sparse_kernel.cu"],
    verbose=True,
    extra_cuda_cflags=["-O2"],
    extra_cflags=["-O2"]
)

print("✅ BLOCK 7 compiled")

# =========================
# TEST
# =========================
B, T, H = 64, 128, 512

x = torch.randn(B, T, H, device="cuda")
W = torch.randn(H // 64, 4, 64, 64, device="cuda")
h = torch.zeros(B, H, device="cuda")
c = torch.zeros(B, H, device="cuda")

out = mod.forward(x, W, h, c)

print("Output shape:", out.shape)

✅ BLOCK 7 compiled
Output shape: torch.Size([64, 128, 512])


In [17]:
# =========================
# BLOCK 8: OPTIMIZED SAFE VERSION
# =========================

from torch.utils.cpp_extension import load
import os, time, torch

os.system("rm -rf v3")
os.system("rm -rf /root/.cache/torch_extensions")
os.makedirs("v3", exist_ok=True)

# =========================
# CUDA KERNEL
# =========================
with open("v3/sparse_kernel.cu", "w") as f:
    f.write(r'''
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

#define BLOCK_SIZE 64
#define TILE_SIZE 32

__device__ inline float sigmoidf(float x) {
    return 1.f / (1.f + expf(-x));
}

__global__ void lstm_kernel(
    const float* x,
    const float* W,
    float* h,
    float* c,
    float* out,
    int B, int T, int H
) {
    int b = blockIdx.x;
    int block_id = blockIdx.y;
    int tid = threadIdx.x;

    int h_idx = block_id * BLOCK_SIZE + tid;

    if (b >= B || h_idx >= H) return;

    __shared__ float W_shared[BLOCK_SIZE][BLOCK_SIZE];

    float h_val = h[b * H + h_idx];
    float c_val = c[b * H + h_idx];

    // --------------------------
    // PARTIAL REGISTER CACHE
    // --------------------------
    float h_tile[TILE_SIZE];

    #pragma unroll
    for (int k = 0; k < TILE_SIZE; k++) {
        h_tile[k] = h[b * H + block_id * BLOCK_SIZE + k];
    }

    for (int t = 0; t < T; t++) {

        float i_val = 0.f, f_val = 0.f, g_val = 0.f, o_val = 0.f;

        for (int g = 0; g < 4; g++) {

            // load weights
            for (int i = tid; i < BLOCK_SIZE * BLOCK_SIZE; i += BLOCK_SIZE) {
                int r = i / BLOCK_SIZE;
                int c_idx = i % BLOCK_SIZE;

                W_shared[r][c_idx] =
                    W[block_id * 4 * BLOCK_SIZE * BLOCK_SIZE +
                      g * BLOCK_SIZE * BLOCK_SIZE +
                      r * BLOCK_SIZE + c_idx];
            }

            __syncthreads();

            float acc = 0.f;

            // --------------------------
            // COMPUTE USING TILE
            // --------------------------
            for (int k = 0; k < TILE_SIZE; k++) {
                acc += W_shared[tid][k] * h_tile[k];
            }

            // remaining half from global
            for (int k = TILE_SIZE; k < BLOCK_SIZE; k++) {
                float h_k = h[b * H + block_id * BLOCK_SIZE + k];
                acc += W_shared[tid][k] * h_k;
            }

            float x_val = x[b * T * H + t * H + h_idx];

            if (g == 0) i_val = sigmoidf(acc + x_val);
            else if (g == 1) f_val = sigmoidf(acc + x_val);
            else if (g == 2) g_val = tanhf(acc + x_val);
            else o_val = sigmoidf(acc + x_val);

            __syncthreads();
        }

        c_val = f_val * c_val + i_val * g_val;
        h_val = o_val * tanhf(c_val);

        // update tile
        if (tid < TILE_SIZE) {
            h_tile[tid] = h_val;
        }

        out[b * T * H + t * H + h_idx] = h_val;
        h[b * H + h_idx] = h_val;
    }

    c[b * H + h_idx] = c_val;
}

void launch_lstm(
    torch::Tensor x,
    torch::Tensor W,
    torch::Tensor h,
    torch::Tensor c,
    torch::Tensor out
) {
    int B = x.size(0);
    int T = x.size(1);
    int H = x.size(2);

    int num_blocks = H / BLOCK_SIZE;

    dim3 grid(B, num_blocks);
    dim3 block(BLOCK_SIZE);

    lstm_kernel<<<grid, block>>>(
        x.data_ptr<float>(),
        W.data_ptr<float>(),
        h.data_ptr<float>(),
        c.data_ptr<float>(),
        out.data_ptr<float>(),
        B, T, H
    );
}
''')

# =========================
# BINDING
# =========================
with open("v3/binding.cpp", "w") as f:
    f.write(r'''
#include <torch/extension.h>

void launch_lstm(
    torch::Tensor x,
    torch::Tensor W,
    torch::Tensor h,
    torch::Tensor c,
    torch::Tensor out
);

torch::Tensor forward(
    torch::Tensor x,
    torch::Tensor W,
    torch::Tensor h,
    torch::Tensor c
) {
    auto out = torch::zeros_like(x);
    launch_lstm(x, W, h, c, out);
    return out;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("forward", &forward);
}
''')

# =========================
# COMPILE
# =========================
name = f"v3_opt_{int(time.time())}"

mod = load(
    name=name,
    sources=["v3/binding.cpp", "v3/sparse_kernel.cu"],
    verbose=True,
    extra_cuda_cflags=["-O2"],
    extra_cflags=["-O2"]
)

print("✅ BLOCK 8 compiled")

# =========================
# TEST
# =========================
B, T, H = 64, 128, 512

x = torch.randn(B, T, H, device="cuda")
W = torch.randn(H // 64, 4, 64, 64, device="cuda")
h = torch.zeros(B, H, device="cuda")
c = torch.zeros(B, H, device="cuda")

out = mod.forward(x, W, h, c)

print("Output shape:", out.shape)

✅ BLOCK 8 compiled
Output shape: torch.Size([64, 128, 512])


In [18]:
# =========================
# BLOCK 9: BENCHMARK
# =========================

import torch

def benchmark_cuda(fn, *args, steps=50, name="kernel"):
    # Warmup
    for _ in range(10):
        fn(*args)

    torch.cuda.synchronize()

    starter = torch.cuda.Event(enable_timing=True)
    ender = torch.cuda.Event(enable_timing=True)

    times = []

    for _ in range(steps):
        starter.record()

        fn(*args)

        ender.record()
        torch.cuda.synchronize()

        times.append(starter.elapsed_time(ender))

    avg = sum(times) / len(times)
    print(f"{name}: {avg:.3f} ms")
    return avg


# =========================
# WRAPPER (IMPORTANT)
# =========================
def run_v3():
    return mod.forward(x, W, h, c)


# =========================
# RUN BENCHMARK
# =========================
print("\n===== V3 TIMING =====")

benchmark_cuda(
    run_v3,
    name="V3 Optimized Sparse LSTM"
)


===== V3 TIMING =====
V3 Optimized Sparse LSTM: 22.265 ms


22.26471237182617

In [19]:
# =========================
# BLOCK 10: V4 PERSISTENT KERNEL
# =========================

from torch.utils.cpp_extension import load
import os, time, torch

os.system("rm -rf v4")
os.system("rm -rf /root/.cache/torch_extensions")
os.makedirs("v4", exist_ok=True)

# =========================
# CUDA KERNEL
# =========================
with open("v4/kernel.cu", "w") as f:
    f.write(r'''
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

#define BLOCK_SIZE 64

__device__ inline float sigmoidf(float x) {
    return 1.f / (1.f + expf(-x));
}

__global__ void persistent_lstm_kernel(
    const float* x,
    const float* W,
    float* h,
    float* c,
    float* out,
    int B, int T, int H
) {
    int b = blockIdx.x;
    int block_id = blockIdx.y;
    int tid = threadIdx.x;

    int h_idx = block_id * BLOCK_SIZE + tid;

    if (b >= B || h_idx >= H) return;

    // --------------------------
    // FULL REGISTER STATE
    // --------------------------
    float h_reg[BLOCK_SIZE];

    #pragma unroll
    for (int k = 0; k < BLOCK_SIZE; k++) {
        h_reg[k] = h[b * H + block_id * BLOCK_SIZE + k];
    }

    float h_val = h_reg[tid];
    float c_val = c[b * H + h_idx];

    // --------------------------
    // SHARED MEMORY (single gate)
    // --------------------------
    __shared__ float W_shared[BLOCK_SIZE][BLOCK_SIZE];

    int total = BLOCK_SIZE * BLOCK_SIZE;

    // --------------------------
    // TIME LOOP (PERSISTENT)
    // --------------------------
    for (int t = 0; t < T; t++) {

        float i_val = 0.f, f_val = 0.f, g_val = 0.f, o_val = 0.f;

        for (int g = 0; g < 4; g++) {

            // load weights
            for (int i = tid; i < total; i += BLOCK_SIZE) {
                int r = i / BLOCK_SIZE;
                int c_idx = i % BLOCK_SIZE;

                W_shared[r][c_idx] =
                    W[block_id * 4 * total +
                      g * total +
                      r * BLOCK_SIZE + c_idx];
            }

            __syncthreads();

            float acc = 0.f;

            #pragma unroll
            for (int k = 0; k < BLOCK_SIZE; k++) {
                acc += W_shared[tid][k] * h_reg[k];
            }

            float x_val = x[b * T * H + t * H + h_idx];

            if (g == 0) i_val = sigmoidf(acc + x_val);
            else if (g == 1) f_val = sigmoidf(acc + x_val);
            else if (g == 2) g_val = tanhf(acc + x_val);
            else o_val = sigmoidf(acc + x_val);

            __syncthreads();
        }

        c_val = f_val * c_val + i_val * g_val;
        h_val = o_val * tanhf(c_val);

        // update FULL register block
        h_reg[tid] = h_val;

        out[b * T * H + t * H + h_idx] = h_val;
    }

    // --------------------------
    // WRITE BACK ONCE
    // --------------------------
    h[b * H + h_idx] = h_val;
    c[b * H + h_idx] = c_val;
}

void launch_persistent(
    torch::Tensor x,
    torch::Tensor W,
    torch::Tensor h,
    torch::Tensor c,
    torch::Tensor out
) {
    int B = x.size(0);
    int H = x.size(2);

    int num_blocks = H / BLOCK_SIZE;

    dim3 grid(B, num_blocks);
    dim3 block(BLOCK_SIZE);

    persistent_lstm_kernel<<<grid, block>>>(
        x.data_ptr<float>(),
        W.data_ptr<float>(),
        h.data_ptr<float>(),
        c.data_ptr<float>(),
        out.data_ptr<float>(),
        B, x.size(1), H
    );
}
''')

# =========================
# BINDING
# =========================
with open("v4/bind.cpp", "w") as f:
    f.write(r'''
#include <torch/extension.h>

void launch_persistent(
    torch::Tensor x,
    torch::Tensor W,
    torch::Tensor h,
    torch::Tensor c,
    torch::Tensor out
);

torch::Tensor forward(
    torch::Tensor x,
    torch::Tensor W,
    torch::Tensor h,
    torch::Tensor c
) {
    auto out = torch::zeros_like(x);
    launch_persistent(x, W, h, c, out);
    return out;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("forward", &forward);
}
''')

# =========================
# COMPILE
# =========================
name = f"v4_{int(time.time())}"

mod_v4 = load(
    name=name,
    sources=["v4/bind.cpp", "v4/kernel.cu"],
    verbose=True,
    extra_cuda_cflags=["-O2"],
    extra_cflags=["-O2"]
)

print("✅ V4 compiled")


✅ V4 compiled


In [20]:
# =========================
# BLOCK 11: V4 BENCHMARK + VALIDATION
# =========================

import torch

# -------------------------
# INPUT (same as before)
# -------------------------
B, T, H = 64, 128, 512

x = torch.randn(B, T, H, device="cuda")
W = torch.randn(H // 64, 4, 64, 64, device="cuda")

h = torch.zeros(B, H, device="cuda")
c = torch.zeros(B, H, device="cuda")

# clone for fair test
h_v4 = h.clone()
c_v4 = c.clone()

# -------------------------
# WRAPPER
# -------------------------
def run_v4():
    return mod_v4.forward(x, W, h_v4, c_v4)

# -------------------------
# BENCHMARK FUNCTION
# -------------------------
def benchmark_cuda(fn, steps=50, name="kernel"):
    # warmup
    for _ in range(10):
        fn()

    torch.cuda.synchronize()

    starter = torch.cuda.Event(enable_timing=True)
    ender = torch.cuda.Event(enable_timing=True)

    times = []

    for _ in range(steps):
        starter.record()
        fn()
        ender.record()

        torch.cuda.synchronize()
        times.append(starter.elapsed_time(ender))

    avg = sum(times) / len(times)
    print(f"{name}: {avg:.3f} ms")
    return avg


# -------------------------
# RUN
# -------------------------
print("\n===== V4 TIMING =====")

t_v4 = benchmark_cuda(run_v4, name="V4 Persistent LSTM")

print("\nSpeedup vs V3 (~22.3 ms):", 22.3 / t_v4)


===== V4 TIMING =====
V4 Persistent LSTM: 17.176 ms

Speedup vs V3 (~22.3 ms): 1.2983502849697737


In [25]:
import os
os.system("rm -rf /root/.cache/torch_extensions")

0

In [27]:
import os

os.system("rm -rf /root/.cache/torch_extensions")
os.system("rm -rf v4_1/build")

0

In [32]:
import os
from torch.utils.cpp_extension import load

# =========================
# CLEAN BUILD
# =========================
os.system("rm -rf v4_1")
os.system("rm -rf /root/.cache/torch_extensions")
os.makedirs("v4_1", exist_ok=True)

# =========================
# CUDA KERNEL (FIXED)
# =========================
with open("v4_1/kernel.cu", "w") as f:
    f.write(r"""
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <math.h>
#include <ATen/cuda/CUDAContext.h>

#define BLOCK_SIZE 64
#define TILE_K 32   // 🔥 FIX: reduces shared memory

__device__ __forceinline__ float sigmoidf(float x) {
    return 1.0f / (1.0f + expf(-x));
}

__launch_bounds__(64, 2)
__global__ void fused_lstm_kernel(
    const float* __restrict__ x,
    const float* __restrict__ W,
    float* __restrict__ h,
    float* __restrict__ c,
    float* __restrict__ out,
    int B, int T, int H
) {
    int b = blockIdx.x;
    int block_id = blockIdx.y;
    int tid = threadIdx.x;

    int h_idx = block_id * BLOCK_SIZE + tid;
    if (b >= B || h_idx >= H) return;

    // persistent hidden state
    float h_reg[BLOCK_SIZE];

    #pragma unroll
    for (int k = 0; k < BLOCK_SIZE; k++) {
        h_reg[k] = h[b * H + block_id * BLOCK_SIZE + k];
    }

    float h_val = h_reg[tid];
    float c_val = c[b * H + h_idx];

    // 🔥 reduced shared memory
    __shared__ float W_shared[BLOCK_SIZE][TILE_K];

    for (int t = 0; t < T; t++) {

        float i_val = 0.f, f_val = 0.f, g_val = 0.f, o_val = 0.f;

        // ---- fused gates ----
        for (int gate = 0; gate < 4; gate++) {

            for (int k0 = 0; k0 < BLOCK_SIZE; k0 += TILE_K) {

                // load tile
                for (int i = tid; i < BLOCK_SIZE * TILE_K; i += BLOCK_SIZE) {
                    int r = i / TILE_K;
                    int c_idx = i % TILE_K;

                    int global_k = k0 + c_idx;

                    W_shared[r][c_idx] =
                        W[block_id * (4 * BLOCK_SIZE * BLOCK_SIZE) +
                          gate * (BLOCK_SIZE * BLOCK_SIZE) +
                          r * BLOCK_SIZE + global_k];
                }

                __syncthreads();

                #pragma unroll
                for (int k = 0; k < TILE_K; k++) {
                    float h_k = h_reg[k0 + k];
                    float w = W_shared[tid][k];

                    if (gate == 0) i_val += w * h_k;
                    else if (gate == 1) f_val += w * h_k;
                    else if (gate == 2) g_val += w * h_k;
                    else o_val += w * h_k;
                }

                __syncthreads();
            }
        }

        float x_val = x[b * T * H + t * H + h_idx];

        i_val = sigmoidf(i_val + x_val);
        f_val = sigmoidf(f_val + x_val);
        g_val = tanhf(g_val + x_val);
        o_val = sigmoidf(o_val + x_val);

        c_val = f_val * c_val + i_val * g_val;
        h_val = o_val * tanhf(c_val);

        h_reg[tid] = h_val;

        out[b * T * H + t * H + h_idx] = h_val;

        __syncthreads();
    }

    h[b * H + h_idx] = h_val;
    c[b * H + h_idx] = c_val;
}

void launch_fused(
    torch::Tensor x,
    torch::Tensor W,
    torch::Tensor h,
    torch::Tensor c,
    torch::Tensor out
) {
    int B = x.size(0);
    int T = x.size(1);
    int H = x.size(2);

    int num_blocks = H / BLOCK_SIZE;

    dim3 grid(B, num_blocks);
    dim3 block(BLOCK_SIZE);

    cudaStream_t stream = at::cuda::getCurrentCUDAStream();

    fused_lstm_kernel<<<grid, block, 0, stream>>>(
        x.data_ptr<float>(),
        W.data_ptr<float>(),
        h.data_ptr<float>(),
        c.data_ptr<float>(),
        out.data_ptr<float>(),
        B, T, H
    );

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess) {
        printf("CUDA ERROR: %s\n", cudaGetErrorString(err));
    }
}
""")

# =========================
# BINDING
# =========================
with open("v4_1/bind.cpp", "w") as f:
    f.write(r"""
#include <torch/extension.h>

void launch_fused(
    torch::Tensor x,
    torch::Tensor W,
    torch::Tensor h,
    torch::Tensor c,
    torch::Tensor out
);

torch::Tensor forward(
    torch::Tensor x,
    torch::Tensor W,
    torch::Tensor h,
    torch::Tensor c
) {
    TORCH_CHECK(x.is_cuda(), "x must be CUDA");
    TORCH_CHECK(W.is_cuda(), "W must be CUDA");
    TORCH_CHECK(h.is_cuda(), "h must be CUDA");
    TORCH_CHECK(c.is_cuda(), "c must be CUDA");

    auto out = torch::zeros_like(x);

    launch_fused(x, W, h, c, out);

    return out;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("forward", &forward, "Fused LSTM Forward");
}
""")

# =========================
# BUILD (FIXED FLAGS)
# =========================
os.environ["MAX_JOBS"] = "1"
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"

mod_v4_1 = load(
    name="v4_1",
    sources=["v4_1/bind.cpp", "v4_1/kernel.cu"],
    verbose=True,
    extra_cuda_cflags=[
        "-O3",
        "--use_fast_math",
        "-maxrregcount=128"
    ],
    extra_cflags=[
        "-O3",
        "-std=c++17",
        "-D_GLIBCXX_USE_CXX11_ABI=1"
    ],
)

print("✅ BUILD SUCCESS")

✅ BUILD SUCCESS


In [33]:
import torch

B, T, H = 2, 4, 64

x = torch.randn(B, T, H, device="cuda")
W = torch.randn(H // 64, 4, 64, 64, device="cuda").reshape(H // 64, 64, 256)
h = torch.randn(B, H, device="cuda")
c = torch.randn(B, H, device="cuda")

out = mod_v4_1.forward(x, W, h, c)

print(out.shape)
print(out.isnan().any())

torch.Size([2, 4, 64])
tensor(False, device='cuda:0')


In [34]:
import torch
import time

torch.cuda.synchronize()

start = time.time()
for _ in range(50):
    out = mod_v4_1.forward(x, W, h, c)
torch.cuda.synchronize()

end = time.time()

print("Avg ms:", (end - start) * 1000 / 50)

Avg ms: 0.24152755737304688


In [35]:
B, T, H = 32, 100, 512

x = torch.randn(B, T, H, device="cuda")
W = torch.randn(H // 64, 4, 64, 64, device="cuda").reshape(H // 64, 64, 256)
h = torch.randn(B, H, device="cuda")
c = torch.randn(B, H, device="cuda")

torch.cuda.synchronize()

start = time.time()
for _ in range(20):
    out = mod_v4_1.forward(x, W, h, c)
torch.cuda.synchronize()

end = time.time()

print("Avg ms:", (end - start) * 1000 / 20)

Avg ms: 10.671389102935791


In [38]:
import os
import torch
import time
from torch.utils.cpp_extension import load

# =========================
# CLEAN BUILD
# =========================
os.system("rm -rf v4_1")
os.system("rm -rf /root/.cache/torch_extensions")
os.makedirs("v4_1", exist_ok=True)

# =========================
# CUDA KERNEL (UNCHANGED & CORRECT)
# =========================
with open("v4_1/kernel.cu", "w") as f:
    f.write(r"""
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <math.h>
#include <ATen/cuda/CUDAContext.h>

#define BLOCK_SIZE 64
#define TILE_K 32

__device__ __forceinline__ float sigmoidf(float x) {
    return 1.0f / (1.0f + expf(-x));
}

__launch_bounds__(64, 2)
__global__ void fused_lstm_kernel(
    const float* __restrict__ Wx,
    const float* __restrict__ W,
    float* __restrict__ h,
    float* __restrict__ c,
    float* __restrict__ out,
    int B, int T, int H
) {
    int b = blockIdx.x;
    int block_id = blockIdx.y;
    int tid = threadIdx.x;

    int h_idx = block_id * BLOCK_SIZE + tid;
    if (b >= B || h_idx >= H) return;

    float h_reg[BLOCK_SIZE];

    #pragma unroll
    for (int k = 0; k < BLOCK_SIZE; k++) {
        h_reg[k] = h[b * H + block_id * BLOCK_SIZE + k];
    }

    float h_val = h_reg[tid];
    float c_val = c[b * H + h_idx];

    __shared__ float W_shared[BLOCK_SIZE][TILE_K];

    for (int t = 0; t < T; t++) {

        float i_val = 0.f, f_val = 0.f, g_val = 0.f, o_val = 0.f;

        for (int gate = 0; gate < 4; gate++) {

            for (int k0 = 0; k0 < BLOCK_SIZE; k0 += TILE_K) {

                for (int i = tid; i < BLOCK_SIZE * TILE_K; i += BLOCK_SIZE) {
                    int r = i / TILE_K;
                    int c_idx = i % TILE_K;
                    int global_k = k0 + c_idx;

                    W_shared[r][c_idx] =
                        W[block_id * (4 * BLOCK_SIZE * BLOCK_SIZE) +
                          gate * (BLOCK_SIZE * BLOCK_SIZE) +
                          r * BLOCK_SIZE + global_k];
                }

                __syncthreads();

                for (int k = 0; k < TILE_K; k++) {
                    float h_k = h_reg[k0 + k];
                    float w = W_shared[tid][k];

                    if (gate == 0) i_val += w * h_k;
                    else if (gate == 1) f_val += w * h_k;
                    else if (gate == 2) g_val += w * h_k;
                    else o_val += w * h_k;
                }

                __syncthreads();
            }
        }

        float x_val = Wx[b * T * H + t * H + h_idx];

        i_val = sigmoidf(i_val + x_val);
        f_val = sigmoidf(f_val + x_val);
        g_val = tanhf(g_val + x_val);
        o_val = sigmoidf(o_val + x_val);

        c_val = f_val * c_val + i_val * g_val;
        h_val = o_val * tanhf(c_val);

        h_reg[tid] = h_val;

        out[b * T * H + t * H + h_idx] = h_val;

        __syncthreads();
    }

    h[b * H + h_idx] = h_val;
    c[b * H + h_idx] = c_val;
}

void launch_fused(
    torch::Tensor Wx,
    torch::Tensor W,
    torch::Tensor h,
    torch::Tensor c,
    torch::Tensor out
) {
    int B = Wx.size(0);
    int T = Wx.size(1);
    int H = Wx.size(2);

    int num_blocks = H / BLOCK_SIZE;

    dim3 grid(B, num_blocks);
    dim3 block(BLOCK_SIZE);

    cudaStream_t stream = at::cuda::getCurrentCUDAStream();

    fused_lstm_kernel<<<grid, block, 0, stream>>>(
        Wx.data_ptr<float>(),
        W.data_ptr<float>(),
        h.data_ptr<float>(),
        c.data_ptr<float>(),
        out.data_ptr<float>(),
        B, T, H
    );
}
""")

# =========================
# BINDING
# =========================
with open("v4_1/bind.cpp", "w") as f:
    f.write(r"""
#include <torch/extension.h>

void launch_fused(
    torch::Tensor Wx,
    torch::Tensor W,
    torch::Tensor h,
    torch::Tensor c,
    torch::Tensor out
);

torch::Tensor forward(
    torch::Tensor Wx,
    torch::Tensor W,
    torch::Tensor h,
    torch::Tensor c
) {
    auto out = torch::zeros_like(Wx);
    launch_fused(Wx, W, h, c, out);
    return out;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("forward", &forward, "Fused LSTM Forward");
}
""")

# =========================
# BUILD
# =========================
os.environ["MAX_JOBS"] = "1"
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"

mod = load(
    name="v4_1",
    sources=["v4_1/bind.cpp", "v4_1/kernel.cu"],
    verbose=False,
    extra_cuda_cflags=["-O3", "--use_fast_math", "-maxrregcount=128"],
    extra_cflags=["-O3", "-std=c++17", "-D_GLIBCXX_USE_CXX11_ABI=1"],
)

print("✅ BUILD SUCCESS")

# =========================
# TEST CONFIG
# =========================
B, T, H = 32, 100, 512
BLOCK = 64
NUM_BLOCKS = H // BLOCK

x = torch.randn(B, T, H, device="cuda")
Wx = torch.randn(B, T, H, device="cuda")  # already projected

h0 = torch.randn(B, H, device="cuda")
c0 = torch.randn(B, H, device="cuda")

# block weights (correct)
W_blocks = torch.randn(NUM_BLOCKS, BLOCK, 4 * BLOCK, device="cuda")

# =========================
# REFERENCE (MATCHES KERNEL EXACTLY)
# =========================
def lstm_ref_block(Wx, W_blocks, h, c):
    B, T, H = Wx.shape
    BLOCK = 64
    NUM_BLOCKS = H // BLOCK

    outputs = []

    for t in range(T):
        h_new = torch.zeros_like(h)
        c_new = torch.zeros_like(c)

        for b in range(NUM_BLOCKS):
            hs = b * BLOCK
            he = hs + BLOCK

            h_block = h[:, hs:he]

            W = W_blocks[b]

            gates = h_block @ W

            i, f, g, o = gates.chunk(4, dim=1)

            x_block = Wx[:, t, hs:he]

            i = torch.sigmoid(i + x_block)
            f = torch.sigmoid(f + x_block)
            g = torch.tanh(g + x_block)
            o = torch.sigmoid(o + x_block)

            c_block = f * c[:, hs:he] + i * g
            h_block = o * torch.tanh(c_block)

            h_new[:, hs:he] = h_block
            c_new[:, hs:he] = c_block

        h, c = h_new, c_new
        outputs.append(h.unsqueeze(1))

    return torch.cat(outputs, dim=1)

# =========================
# CORRECTNESS
# =========================
torch.cuda.synchronize()
out_ref = lstm_ref_block(Wx, W_blocks, h0.clone(), c0.clone())
torch.cuda.synchronize()

torch.cuda.synchronize()
out_kernel = mod.forward(Wx, W_blocks, h0.clone(), c0.clone())
torch.cuda.synchronize()

diff = (out_ref - out_kernel).abs().mean().item()
print("Mean abs diff:", diff)

# =========================
# BENCHMARK
# =========================
torch.cuda.synchronize()
start = time.time()
for _ in range(10):
    out = mod.forward(Wx, W_blocks, h0, c0)
torch.cuda.synchronize()
end = time.time()

print("Kernel ms:", (end - start) * 1000 / 10)

torch.cuda.synchronize()
start = time.time()
for _ in range(10):
    out = lstm_ref_block(Wx, W_blocks, h0, c0)
torch.cuda.synchronize()
end = time.time()

print("Reference ms:", (end - start) * 1000 / 10)

✅ BUILD SUCCESS
Mean abs diff: 0.4352041482925415
Kernel ms: 12.712430953979492
Reference ms: 204.5922040939331


In [40]:
import os
from torch.utils.cpp_extension import load

# CLEAN
os.system("rm -rf v4_1")
os.system("rm -rf /root/.cache/torch_extensions")
os.makedirs("v4_1", exist_ok=True)

# =========================
# CUDA KERNEL
# =========================
with open("v4_1/kernel.cu", "w") as f:
    f.write(r"""
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <math.h>
#include <ATen/cuda/CUDAContext.h>

#define BLOCK_SIZE 64
#define TILE_K 32

__device__ __forceinline__ float sigmoidf(float x) {
    return 1.0f / (1.0f + expf(-x));
}

__launch_bounds__(64, 2)
__global__ void fused_lstm_kernel(
    const float* __restrict__ Wx,   // (B, T, 4H)
    const float* __restrict__ W,
    float* __restrict__ h,
    float* __restrict__ c,
    float* __restrict__ out,
    int B, int T, int H
) {
    int b = blockIdx.x;
    int block_id = blockIdx.y;
    int tid = threadIdx.x;

    int h_idx = block_id * BLOCK_SIZE + tid;
    if (b >= B || h_idx >= H) return;

    float h_reg[BLOCK_SIZE];

    #pragma unroll
    for (int k = 0; k < BLOCK_SIZE; k++) {
        h_reg[k] = h[b * H + block_id * BLOCK_SIZE + k];
    }

    float h_val = h_reg[tid];
    float c_val = c[b * H + h_idx];

    __shared__ float W_shared[BLOCK_SIZE][TILE_K];

    for (int t = 0; t < T; t++) {

        float i_val = 0.f, f_val = 0.f, g_val = 0.f, o_val = 0.f;

        for (int gate = 0; gate < 4; gate++) {

            for (int k0 = 0; k0 < BLOCK_SIZE; k0 += TILE_K) {

                for (int i = tid; i < BLOCK_SIZE * TILE_K; i += BLOCK_SIZE) {
                    int r = i / TILE_K;
                    int c_idx = i % TILE_K;
                    int global_k = k0 + c_idx;

                    W_shared[r][c_idx] =
                        W[block_id * (4 * BLOCK_SIZE * BLOCK_SIZE) +
                          gate * (BLOCK_SIZE * BLOCK_SIZE) +
                          r * BLOCK_SIZE + global_k];
                }

                __syncthreads();

                for (int k = 0; k < TILE_K; k++) {
                    float h_k = h_reg[k0 + k];
                    float w = W_shared[tid][k];

                    if (gate == 0) i_val += w * h_k;
                    else if (gate == 1) f_val += w * h_k;
                    else if (gate == 2) g_val += w * h_k;
                    else o_val += w * h_k;
                }

                __syncthreads();
            }
        }

        // ✅ CORRECT Wx (4H)
        int base = b * T * (4 * H) + t * (4 * H) + block_id * (4 * BLOCK_SIZE) + tid;

        float xi = Wx[base + 0 * BLOCK_SIZE];
        float xf = Wx[base + 1 * BLOCK_SIZE];
        float xg = Wx[base + 2 * BLOCK_SIZE];
        float xo = Wx[base + 3 * BLOCK_SIZE];

        i_val = sigmoidf(i_val + xi);
        f_val = sigmoidf(f_val + xf);
        g_val = tanhf(g_val + xg);
        o_val = sigmoidf(o_val + xo);

        c_val = f_val * c_val + i_val * g_val;
        h_val = o_val * tanhf(c_val);

        h_reg[tid] = h_val;

        out[b * T * H + t * H + h_idx] = h_val;

        __syncthreads();
    }

    h[b * H + h_idx] = h_val;
    c[b * H + h_idx] = c_val;
}

void launch_fused(
    torch::Tensor Wx,
    torch::Tensor W,
    torch::Tensor h,
    torch::Tensor c,
    torch::Tensor out
) {
    int B = Wx.size(0);
    int T = Wx.size(1);
    int H = h.size(1);

    int num_blocks = H / BLOCK_SIZE;

    dim3 grid(B, num_blocks);
    dim3 block(BLOCK_SIZE);

    cudaStream_t stream = at::cuda::getCurrentCUDAStream();

    fused_lstm_kernel<<<grid, block, 0, stream>>>(
        Wx.data_ptr<float>(),
        W.data_ptr<float>(),
        h.data_ptr<float>(),
        c.data_ptr<float>(),
        out.data_ptr<float>(),
        B, T, H
    );
}
""")

# =========================
# BINDING
# =========================
with open("v4_1/bind.cpp", "w") as f:
    f.write(r"""
#include <torch/extension.h>

void launch_fused(
    torch::Tensor Wx,
    torch::Tensor W,
    torch::Tensor h,
    torch::Tensor c,
    torch::Tensor out
);

torch::Tensor forward(
    torch::Tensor Wx,
    torch::Tensor W,
    torch::Tensor h,
    torch::Tensor c
) {
    auto out = torch::zeros({Wx.size(0), Wx.size(1), h.size(1)}, Wx.options());
    launch_fused(Wx, W, h, c, out);
    return out;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("forward", &forward);
}
""")

# =========================
# BUILD
# =========================
os.environ["MAX_JOBS"] = "1"
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"

mod = load(
    name="v4_1",
    sources=["v4_1/bind.cpp", "v4_1/kernel.cu"],
    verbose=False,
    extra_cuda_cflags=["-O3", "--use_fast_math", "-maxrregcount=128"],
    extra_cflags=["-O3", "-std=c++17", "-D_GLIBCXX_USE_CXX11_ABI=1"],
)

print("✅ BUILD SUCCESS")

✅ BUILD SUCCESS


In [41]:
import torch

B, T, H = 32, 100, 512
BLOCK = 64
NUM_BLOCKS = H // BLOCK

# ✅ FULL Wx (4H)
Wx = torch.randn(B, T, 4 * H, device="cuda")

h0 = torch.randn(B, H, device="cuda")
c0 = torch.randn(B, H, device="cuda")

# block weights
W_blocks = torch.randn(NUM_BLOCKS, BLOCK, 4 * BLOCK, device="cuda")

# =========================
# CORRECT REFERENCE
# =========================
def lstm_ref_block(Wx, W_blocks, h, c):
    B, T, _ = Wx.shape
    H = h.size(1)
    BLOCK = 64
    NUM_BLOCKS = H // BLOCK

    outputs = []

    for t in range(T):
        h_new = torch.zeros_like(h)
        c_new = torch.zeros_like(c)

        for b in range(NUM_BLOCKS):
            hs = b * BLOCK
            he = hs + BLOCK

            h_block = h[:, hs:he]
            W = W_blocks[b]

            gates = h_block @ W
            i, f, g, o = gates.chunk(4, dim=1)

            x_block = Wx[:, t, hs*4:(hs*4 + 4*BLOCK)]
            xi, xf, xg, xo = x_block.chunk(4, dim=1)

            i = torch.sigmoid(i + xi)
            f = torch.sigmoid(f + xf)
            g = torch.tanh(g + xg)
            o = torch.sigmoid(o + xo)

            c_block = f * c[:, hs:he] + i * g
            h_block = o * torch.tanh(c_block)

            h_new[:, hs:he] = h_block
            c_new[:, hs:he] = c_block

        h, c = h_new, c_new
        outputs.append(h.unsqueeze(1))

    return torch.cat(outputs, dim=1)

In [42]:
import time

# correctness
torch.cuda.synchronize()
out_ref = lstm_ref_block(Wx, W_blocks, h0.clone(), c0.clone())
torch.cuda.synchronize()

torch.cuda.synchronize()
out_kernel = mod.forward(Wx, W_blocks, h0.clone(), c0.clone())
torch.cuda.synchronize()

diff = (out_ref - out_kernel).abs().mean().item()
print("Mean abs diff:", diff)

# benchmark kernel
torch.cuda.synchronize()
start = time.time()
for _ in range(10):
    out = mod.forward(Wx, W_blocks, h0, c0)
torch.cuda.synchronize()
end = time.time()

print("Kernel ms:", (end - start) * 1000 / 10)

# benchmark reference
torch.cuda.synchronize()
start = time.time()
for _ in range(10):
    out = lstm_ref_block(Wx, W_blocks, h0, c0)
torch.cuda.synchronize()
end = time.time()

print("Reference ms:", (end - start) * 1000 / 10)

Mean abs diff: 0.43435028195381165
Kernel ms: 11.301183700561523
Reference ms: 311.33501529693604
